# Real Estate Price Prediction: Data Cleaning & Modeling
This notebook demonstrates the complete machine learning pipeline for building a Real Estate Price Prediction model using the **Bengaluru House Price Dataset**. 

We will perform the following steps:
1. **Data Loading & Exploration**: Understand the dataset structure.
2. **Data Cleaning**: Handle missing values, parse sizes, and clean numeric features.
3. **Feature Engineering**: Standardize spatial and sizing metrics.
4. **Outlier Detection & Removal**: Use statistical thresholds to remove anomalies.
5. **One-Hot Encoding**: Represent categorical variables (locations) numerically.
6. **Model Selection & Training**: Train a Multiple Linear Regression model.
7. **Evaluation**: Assess model accuracy and write serving artifacts.

### Step 1: Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import json
import os

# Set styling for plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load the dataset
df = pd.read_csv("Bangalore  house data.csv")
print(f"Dataset shape: {df.shape}")
df.head()

### Step 2: Data Cleaning
Let's inspect columns and drop features that aren't critical or have too many missing values to be useful for a linear model.

In [ ]:
# Inspect value counts of area_type
df['area_type'].value_counts()

In [ ]:
# Drop less relevant columns for model simplicity
df2 = df.drop(['area_type', 'society', 'balcony', 'availability'], axis='columns')
df2.head()

In [ ]:
# Check for missing values
df2.isnull().sum()

In [ ]:
# Drop rows with missing values
df3 = df2.dropna()
df3.isnull().sum()

#### Parsing the 'size' column
The `size` column contains string values like "2 BHK" or "4 Bedroom". We'll extract the integer count and label it as `bhk`.

In [ ]:
df3['bhk'] = df3['size'].apply(lambda x: int(x.split(' ')[0]))
df3.head()

#### Cleaning the 'total_sqft' column
Some values are simple floats, while others are ranges (e.g. "1045 - 1105"), or strings with units (e.g. "34.46Sq. Yards"). We will write a function to handle ranges by taking their average and convert valid values to floats.

In [ ]:
def convert_sqft_to_num(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        try:
            return (float(tokens[0]) + float(tokens[1])) / 2
            
        except:
            return None
    try:
        return float(x)
    except:
        return None

In [ ]:
df4 = df3.copy()
df4['total_sqft'] = df4['total_sqft'].apply(convert_sqft_to_num)
# Drop any rows where total_sqft couldn't be parsed
df4 = df4.dropna(subset=['total_sqft'])
df4.shape

### Step 3: Outlier Detection and Removal
Outliers can heavily bias a linear regression model. We will use simple domain rules and statistical metrics to clean the dataset.

#### 3.1 Area per bedroom ratio
In real estate, a bedroom normally requires at least 300 square feet of area. Houses violating this threshold (e.g. a 4 BHK of 600 sqft) are highly likely to be data entry errors or extreme outliers.

In [ ]:
# Inspect houses violating the 300 sqft/bedroom threshold
df4[df4.total_sqft / df4.bhk < 300].head()

In [ ]:
# Filter out these rows
df5 = df4[~(df4.total_sqft/df4.bhk < 300)]
df5.shape

#### 3.2 Price per square foot outliers
Let's create a temporary feature `price_per_sqft` (Price in Lakhs, so `(Price * 100,000) / total_sqft`) and look at its distribution.

In [ ]:
df5['price_per_sqft'] = (df5['price'] * 100000) / df5['total_sqft']
df5['price_per_sqft'].describe()

We see a massive range (min 267 to max 1.76e7). We should filter out properties whose price per square foot is beyond 1 standard deviation from the mean price per square foot in their respective locations.

In [ ]:
def remove_pps_outliers(df_in):
    df_out = pd.DataFrame()
    for key, subdf in df_in.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

df6 = remove_pps_outliers(df5)
df6.shape

#### 3.3 BHK price consistency
Let's check if the price of, say, a 3 BHK house is higher than a 2 BHK house in the same location for a similar area. We will remove cases where an n-BHK house is cheaper than the mean price of an (n-1)-BHK house in that location.

In [ ]:
def remove_bhk_outliers(df_in):
    exclude_indices = np.array([])
    for location, location_df in df_in.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('bhk'):
            bhk_stats[bhk] = {
                'mean': np.mean(bhk_df.price_per_sqft),
                'std': np.std(bhk_df.price_per_sqft),
                'count': bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('bhk'):
            stats = bhk_stats.get(bhk - 1)
            if stats and stats['count'] > 5:
                exclude_indices = np.append(exclude_indices, bhk_df[bhk_df.price_per_sqft < stats['mean']].index.values)
    return df_in.drop(exclude_indices, axis='index')

df7 = remove_bhk_outliers(df6)
df7.shape

#### 3.4 Bathroom outliers
Normally, a house should not have bathrooms exceeding the number of bedrooms + 2. We'll filter these out.

In [ ]:
df8 = df7[df7.bath < df7.bhk + 2]
df8.shape

### Step 4: Categorical Encoding (Location)
There are too many distinct locations. Let's strip whitespace and group locations with less than 30 occurrences into a generic category named `'other'`.

In [ ]:
df8.location = df8.location.apply(lambda x: x.strip())
location_stats = df8.groupby('location')['location'].agg('count').sort_values(ascending=False)
location_stats_less_than_30 = location_stats[location_stats < 30]

df8.location = df8.location.apply(lambda x: 'other' if x in location_stats_less_than_30 else x)
print(f"Unique locations: {df8.location.nunique()}")

In [ ]:
# Create dummies and drop 'other' column to avoid multicollinearity
dummies = pd.get_dummies(df8.location, dtype=int)
if 'other' in dummies.columns:
    dummies = dummies.drop(['other'], axis='columns')

df9 = pd.concat([df8, dummies], axis='columns')
df10 = df9.drop(['size', 'location', 'price_per_sqft'], axis='columns', errors='ignore')
df10.head()

### Step 5: Modeling
We split the data into training and test sets, fit a Multiple Linear Regression model, and evaluate.

In [ ]:
X = df10.drop(['price'], axis='columns')
y = df10['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)
print(f"X_train shape: {X_train.shape}")

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
print(f"Model R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Model MSE: {mean_squared_error(y_test, y_pred):.4f}")

### Step 6: Test Prediction Function

In [ ]:
def predict_price(location, sqft, bath, bhk):
    loc_index = -1
    if location in X.columns:
        loc_index = X.columns.get_loc(location)
        
    x = np.zeros(len(X.columns))
    x[0] = sqft
    x[1] = bath
    x[2] = bhk
    if loc_index >= 0:
        x[loc_index] = 1
        
    return lr.predict([x])[0]

# Examples
print(f"Price for 2 BHK in Indiranagar: {predict_price('Indiranagar', 1000, 2, 2):.2f} Lakhs")
print(f"Price for 3 BHK in 1st Phase JP Nagar: {predict_price('1st Phase JP Nagar', 1500, 3, 3):.2f} Lakhs")

### Step 7: Export Artifacts
Save the model using `joblib` and export coefficients/labels metadata in JSON format.

In [ ]:
# Export model to finished project artifacts
finished_artifacts_dir = "../finished/backend/artifacts"
os.makedirs(finished_artifacts_dir, exist_ok=True)

joblib.dump(lr, os.path.join(finished_artifacts_dir, "model.joblib"))

coefficients_dict = {col: float(coef) for col, coef in zip(X.columns, lr.coef_)}
metadata = {
    "features": list(X.columns),
    "locations": [loc for loc in sorted(list(df8.location.unique())) if loc != 'other'],
    "metrics": {
        "r2_score": float(r2_score(y_test, y_pred)),
        "mse": float(mean_squared_error(y_test, y_pred))
    },
    "intercept": float(lr.intercept_),
    "coefficients": coefficients_dict
}

with open(os.path.join(finished_artifacts_dir, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("Finished exporting model and metadata!")